# PyTorch BYOC パイプライン

このノートブックはカスタムコンテナ (BYOC: Bring Your Own Container) を使用した PyTorch モデルの学習・評価・Pipeline 実行を一気通貫で行います。

### BYOC とは

BYOC (Bring Your Own Container) は、自分でビルドした Docker イメージを SageMaker の Training Job / Processing Job で使用する方式です。AWS マネージドの DLC (Deep Learning Container) にないライブラリが必要な場合や、C 拡張のビルドが必要な場合に使用します。

### DLC 版との違い

| 項目 | DLC 版 (`pytorch-pipeline.ipynb`) | BYOC 版 (このノートブック) |
|------|----------------------------------|---------------------------|
| コンテナ | AWS マネージド PyTorch DLC | ECR 上のカスタムイメージ |
| Docker ビルド | 不要 | 必要 (Section 7) |
| Estimator | `PyTorchEstimator` | 汎用 `Estimator` + `image_uri` |
| Processor | `PyTorchProcessor` | `ScriptProcessor` + `image_uri` |
| 追加ライブラリ | `requirements.txt` で実行時インストール | Dockerfile でビルド時にインストール済み |
| 起動速度 | pip install 分遅い | ビルド済みのため速い |

### このノートブックの流れ

- **Section 1-2**: セットアップとデータ確認
- **Section 3-5**: モデル定義・ローカル学習・ローカル評価
- **Section 6**: MLflow 記録
- **Section 7**: Docker Build & ECR Push
- **Section 8**: SageMaker Training Job (BYOC)
- **Section 9-10**: SageMaker Processing Job (DLC / BYOC)
- **Section 11**: SageMaker Pipeline (Train → Register → Evaluate)

ローカル学習部分 (Section 3-5) は `pytorch-pipeline.ipynb` (DLC 版) と同一です。

### 前提条件

サンプルデータセットは `deploy.sh` の実行時に S3 に自動アップロード済みです。そのまま Notebook を実行できます。

データを変更する場合は `pipelines/container-pytorch-dlc-byoc/data/` の CSV ファイルを更新し、以下のコマンドで再アップロードしてください。

```bash
./pipelines/scripts/01-upload-dataset.sh -c container-pytorch-dlc-byoc
```


---
## 1. セットアップ

学習・評価に必要なライブラリをインストールします。

In [ ]:
# 依存パッケージのインストール
# numpy<2: PyTorch が NumPy 1.x でコンパイルされているため
# torch: CPU 版で再インストール (GPU 版の triton 依存を除去)
!pip install -U "numpy<2" scikit-learn pandas mlflow sagemaker-mlflow matplotlib seaborn
!pip uninstall -y triton triton-nightly 2>/dev/null; pip install --force-reinstall torch --index-url https://download.pytorch.org/whl/cpu


In [ ]:
import os
import io
import json
import logging
import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)
logging.getLogger("sagemaker.jumpstart").setLevel(logging.CRITICAL)

In [ ]:
# === プロジェクト設定 ===
PROJECT_NAME = "sagemaker-ai-ml-pipeline"
STACK_NAME   = "sagemaker-ai-ml-pipeline-stack"
REGION       = boto3.session.Session().region_name or "us-east-1"
CONTAINER_DIR = "../pipelines/container-pytorch-dlc-byoc"

cfn = boto3.client("cloudformation", region_name=REGION)
outputs = {
    o["OutputKey"]: o["OutputValue"]
    for o in cfn.describe_stacks(StackName=STACK_NAME)["Stacks"][0]["Outputs"]
}
DATASET_BUCKET = outputs["DatasetBucketName"]
MODEL_BUCKET   = outputs["ModelArtifactBucketName"]

# S3 パス設定 (01-upload-dataset.sh のアップロード先と合わせる)
CONTAINER_TAG = os.path.basename(CONTAINER_DIR)
S3_TRAIN_PREFIX = f"{CONTAINER_TAG}/train"
S3_TEST_PREFIX  = f"{CONTAINER_TAG}/test"

MODEL_OUTPUT_DIR = "output/model"
EVAL_OUTPUT_DIR  = "/tmp/notebook-pytorch-eval"
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
os.makedirs(EVAL_OUTPUT_DIR, exist_ok=True)

print(f"Dataset bucket : {DATASET_BUCKET}")
print(f"Model bucket   : {MODEL_BUCKET}")

---
## 2. データの読み込みと確認

In [ ]:
s3 = boto3.client("s3", region_name=REGION)

def load_csv_from_s3(bucket, prefix):
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix + "/")
    keys = [obj["Key"] for obj in resp.get("Contents", []) if obj["Key"].endswith(".csv")]
    if not keys:
        raise FileNotFoundError(f"No CSV files found in s3://{bucket}/{prefix}/. Run 01-upload-dataset.sh first.")
    print(f"Found {len(keys)} CSV file(s) in s3://{bucket}/{prefix}/")
    dfs = []
    for key in keys:
        obj = s3.get_object(Bucket=bucket, Key=key)
        dfs.append(pd.read_csv(io.BytesIO(obj["Body"].read())))
    return pd.concat(dfs, ignore_index=True)

df_train = load_csv_from_s3(DATASET_BUCKET, S3_TRAIN_PREFIX)
df_test  = load_csv_from_s3(DATASET_BUCKET, S3_TEST_PREFIX)

print(f"\nTrain shape: {df_train.shape}")
print(f"Test shape:  {df_test.shape}")
df_train.head()

In [ ]:
X_train = df_train.iloc[:, :-1].values.astype(np.float32)
y_train = df_train.iloc[:, -1].values.astype(np.int64)
X_test_np = df_test.iloc[:, :-1].values.astype(np.float32)
y_test = df_test.iloc[:, -1].values.astype(np.int64)
X_test = torch.from_numpy(X_test_np)

input_dim = X_train.shape[1]
num_classes = len(np.unique(y_train))
print(f"Input dim: {input_dim}, Num classes: {num_classes}")
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

---
## 3. モデル定義

`train.py` と同じ `SimpleClassifier` アーキテクチャを定義します。

In [ ]:
class SimpleClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, num_classes),
        )
    def forward(self, x):
        return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleClassifier(input_dim, num_classes).to(device)
print(f"Using device: {device}")
print(model)

---
## 4. ローカル学習

In [ ]:
params = {"epochs": 20, "batch_size": 32, "learning_rate": 0.001}

dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
loader = DataLoader(dataset, batch_size=params["batch_size"], shuffle=True)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=params["learning_rate"])

model.train()
for epoch in range(params["epochs"]):
    total_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        out = model(X_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
        correct += (out.argmax(dim=1) == y_batch).sum().item()
        total += X_batch.size(0)
    avg_loss = total_loss / total
    accuracy = correct / total
    print(f"Epoch {epoch+1}/{params['epochs']} - loss: {avg_loss:.4f}, accuracy: {accuracy:.4f}")

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in loader:
        preds = model(X_batch.to(device)).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())
all_preds, all_labels = np.array(all_preds), np.array(all_labels)
train_accuracy = (all_preds == all_labels).mean()
train_precision = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
train_recall = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
train_f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
print(f"Train accuracy: {train_accuracy:.4f}, precision: {train_precision:.4f}, recall: {train_recall:.4f}, f1: {train_f1:.4f}")

In [ ]:
model_path = os.path.join(MODEL_OUTPUT_DIR, "model.pth")
torch.save({"model_state_dict": model.state_dict(), "input_dim": input_dim, "num_classes": num_classes}, model_path)
S3_MODEL_KEY = "notebook/pytorch/model.pth"
s3.upload_file(model_path, MODEL_BUCKET, S3_MODEL_KEY)
print(f"Model uploaded to s3://{MODEL_BUCKET}/{S3_MODEL_KEY}")

---
## 5. ローカル評価

In [ ]:
model.eval()
with torch.no_grad():
    y_pred = model(X_test.to(device)).argmax(dim=1).cpu().numpy()

metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, average="weighted", zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, average="weighted", zero_division=0)),
    "f1": float(f1_score(y_test, y_pred, average="weighted", zero_division=0)),
}
print("Evaluation Metrics:")
print(json.dumps(metrics, indent=2))

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
labels = sorted(np.unique(y_test))
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title('Confusion Matrix')
plt.tight_layout(); plt.show()

In [ ]:
eval_path = os.path.join(EVAL_OUTPUT_DIR, "evaluation.json")
with open(eval_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Evaluation report saved to {eval_path}")

---
## 6. MLflow 記録

MLflow App に学習・評価のメトリクスをまとめて記録します。CloudFormation スタックで MLflow App がデプロイされていない場合は自動的にスキップされます。

### 記録される情報

| 種類 | 内容 |
|------|------|
| パラメータ | `epochs`, `batch_size`, `learning_rate` |
| メトリクス | `train_accuracy`, `train_loss`, テスト評価メトリクス |
| アーティファクト | 学習済みモデル (`pytorch-byoc-model`)、評価レポート (`evaluation.json`) |

In [ ]:
mlflow_app_arn = ""
mlflow_app_url = ""
try:
    cfn_client = boto3.client("cloudformation", region_name=REGION)
    resp = cfn_client.describe_stacks(StackName=f"{PROJECT_NAME}-stack")
    for output in resp["Stacks"][0]["Outputs"]:
        if output["OutputKey"] == "MlflowAppArn":
            mlflow_app_arn = output["OutputValue"]
            mlflow_app_url = f"https://{mlflow_app_arn.split('/')[-1]}.mlflow.sagemaker.{REGION}.amazonaws.com"
            break
    if mlflow_app_arn:
        print(f"MLflow ARN: {mlflow_app_arn}")
        print(f"MLflow URL: {mlflow_app_url}")
except Exception as e:
    print(f"MLflow App not found ({e}). Skipping MLflow logging.")

In [ ]:
if mlflow_app_arn:
    import mlflow
    from mlflow.models import infer_signature
    logging.getLogger("mlflow.pytorch").setLevel(logging.CRITICAL)
    mlflow.set_tracking_uri(mlflow_app_arn)
    mlflow.set_experiment("pytorch-byoc-notebook")
    with mlflow.start_run():
        mlflow.log_params(params)
        mlflow.log_metric("train_accuracy", train_accuracy)
        mlflow.log_metric("train_loss", avg_loss)
        mlflow.log_metrics(metrics)
        sample_input = torch.from_numpy(X_train[:5]).to(device)
        with torch.no_grad():
            sample_output = model(sample_input).cpu().numpy()
        signature = infer_signature(X_train[:5], sample_output)
        mlflow.pytorch.log_model(pytorch_model=model, artifact_path="pytorch-byoc-model", signature=signature)
        mlflow.log_artifact(eval_path)
    print("Metrics logged to MLflow.")
else:
    print("Skipped (no MLflow App).")

---
## 7. Docker Build & ECR Push

BYOC ではカスタム Docker イメージを ECR にプッシュする必要があります。

### Dockerfile の構成

Dockerfile は `CONTAINER_DIR` 内にあり、AWS マネージドの PyTorch DLC をベースイメージとして使用しています。DLC をベースにすることで、PyTorch・CUDA・cuDNN 等の GPU ライブラリが事前にインストールされた状態から始められます。

### ECR リポジトリ

イメージは `{account_id}.dkr.ecr.{region}.amazonaws.com/{project_name}:{container_tag}` にプッシュされます。リポジトリが存在しない場合は自動作成されます。

In [ ]:
import sagemaker
from sagemaker.inputs import TrainingInput

sm_session = sagemaker.Session(boto_session=boto3.Session(region_name=REGION))
role = sagemaker.get_execution_role()
account_id = boto3.client("sts").get_caller_identity()["Account"]

container_tag = "container-pytorch-dlc-byoc"
ecr_repo = f"{PROJECT_NAME}-container"
ecr_image_uri = f"{account_id}.dkr.ecr.{REGION}.amazonaws.com/{ecr_repo}:{container_tag}"

print(f"SageMaker role: {role}")
print(f"ECR image URI:  {ecr_image_uri}")

ECR にログインし、Docker イメージをビルド・プッシュします。

In [ ]:
import subprocess

ecr_client = boto3.client("ecr", region_name=REGION)
image_exists = False
try:
    ecr_client.describe_images(
        repositoryName=f"{PROJECT_NAME}-container",
        imageIds=[{"imageTag": container_tag}],
    )
    image_exists = True
    print("ECR image already exists. Skipping build.")
    print(f"  {ecr_image_uri}")
except ecr_client.exceptions.ImageNotFoundException:
    print("ECR image not found. Building and pushing...")
except Exception as e:
    print(f"ECR check failed ({e}). Attempting build...")

if not image_exists:
    result = subprocess.run(
        ["bash", "../pipelines/scripts/02-build-and-push-container.sh",
         "-c", container_tag, "--auto-approve", PROJECT_NAME],
        text=True,
        env={**os.environ, "AWS_DEFAULT_REGION": REGION},
    )
    if result.returncode != 0:
        print(f"\nBuild failed (exit code {result.returncode}):")
    else:
        print(f"\nContainer pushed to {ecr_image_uri}")

---
## 8. SageMaker Training Job (BYOC)

カスタムコンテナを使って SageMaker Training Job を実行します。

### DLC 版との違い

| 項目 | DLC 版 (`pytorch-pipeline.ipynb`) | BYOC 版 (このノートブック) |
|------|----------------------------------|---------------------------|
| Estimator | `PyTorchEstimator` | 汎用 `Estimator` |
| コンテナ指定 | `framework_version` で自動選択 | `image_uri` で ECR イメージを直接指定 |
| 追加ライブラリ | `requirements.txt` で実行時インストール | Dockerfile でビルド時にインストール済み |

BYOC を使うメリットは、コンテナ起動時の pip install が不要なため起動が速いこと、および DLC にないライブラリ (C 拡張など) を事前にビルドできることです。

In [ ]:
from sagemaker.estimator import Estimator

byoc_estimator = Estimator(
    image_uri=ecr_image_uri,
    entry_point="train.py",
    source_dir=CONTAINER_DIR,
    role=role,
    instance_type="ml.c7i.xlarge",
    instance_count=1,
    output_path=f"s3://{MODEL_BUCKET}/sagemaker-jobs/pytorch-byoc",
    hyperparameters={"epochs": 20, "batch-size": 32, "learning-rate": 0.001},
    environment={"MLFLOW_APP_ARN": mlflow_app_arn, "MLFLOW_APP_URL": mlflow_app_url},
    sagemaker_session=sm_session,
)

print("BYOC Estimator defined.")

In [ ]:
byoc_estimator.fit(
    # input_mode: "FastFile" = S3 からオンデマンドストリーミング / "File" = 全量ダウンロード後に学習開始
    inputs={"train": TrainingInput(s3_data=f"s3://{DATASET_BUCKET}/{S3_TRAIN_PREFIX}/", input_mode="FastFile")},
    wait=True,
    logs="All",
)

print(f"Training job name: {byoc_estimator.latest_training_job.name}")
print(f"Model artifacts:   {byoc_estimator.model_data}")

---
## 9. SageMaker Processing Job (DLC)

`PyTorchProcessor` を使って DLC マネージドコンテナで評価ジョブを実行します。

### DLC で評価する理由

BYOC コンテナで学習したモデルでも、評価は DLC で実行できます。`evaluate.py` は PyTorch と scikit-learn のみに依存しており、BYOC 固有のライブラリは不要なためです。DLC を使うことでコンテナビルドの手間を省けます。

### 入出力のマッピング

| 入出力 | S3 パス | コンテナ内パス |
|--------|---------|---------------|
| モデル (入力) | Training Job の `model.tar.gz` | `/opt/ml/processing/model` |
| テストデータ (入力) | `s3://{bucket}/container-pytorch-dlc-byoc/test/` | `/opt/ml/processing/test` |
| 評価結果 (出力) | `s3://{bucket}/notebook-pytorch/` | `/opt/ml/processing/evaluation` |

In [ ]:
from sagemaker.pytorch.processing import PyTorchProcessor
from sagemaker.s3 import S3Downloader

# BYOC Training Job のモデルアーティファクトを直接取得
MODEL_S3_URI = byoc_estimator.model_data
print(f"Latest Training Job: {byoc_estimator.latest_training_job.name}")

EVAL_S3_URI = f"s3://{PROJECT_NAME}-eval-{account_id}-{REGION}/notebook-pytorch"
print(f"Model S3:       {MODEL_S3_URI}")
print(f"Eval output S3: {EVAL_S3_URI}")

In [ ]:
eval_processor = PyTorchProcessor(
    framework_version="2.5.1", py_version="py311",
    role=role, instance_count=1, instance_type="ml.c7i.xlarge",
    env={"MLFLOW_APP_ARN": mlflow_app_arn},
    sagemaker_session=sm_session,
)

eval_processor.run(
    code="evaluate.py", source_dir=CONTAINER_DIR,
    inputs=[
        sagemaker.processing.ProcessingInput(source=MODEL_S3_URI, destination="/opt/ml/processing/model"),
        sagemaker.processing.ProcessingInput(source=f"s3://{DATASET_BUCKET}/{S3_TEST_PREFIX}", destination="/opt/ml/processing/test"),
    ],
    outputs=[
        sagemaker.processing.ProcessingOutput(output_name="evaluation", source="/opt/ml/processing/evaluation", destination=EVAL_S3_URI),
    ],
    wait=True, logs=True,
)
print(f"Processing job completed. Results saved to {EVAL_S3_URI}")

In [ ]:
eval_files = S3Downloader.list(EVAL_S3_URI)
for s3_uri in eval_files:
    if s3_uri.endswith("evaluation.json"):
        content = S3Downloader.read_file(s3_uri)
        print("DLC Processing Job - Evaluation Metrics:")
        print(json.dumps(json.loads(content), indent=2))
        break
else:
    print("evaluation.json not found.")

---
## 10. SageMaker Processing Job (BYOC)

カスタムコンテナを使って Processing Job を実行します。`PyTorchProcessor` ではなく `ScriptProcessor` を使用し、`image_uri` でカスタムコンテナを指定します。

DLC 版 (Section 9) と同じ `evaluate.py` を実行しますが、コンテナ内の依存パッケージが BYOC の Dockerfile で定義されたものになります。BYOC 固有のライブラリが評価に必要な場合はこちらを使用してください。

In [ ]:
from sagemaker.processing import ScriptProcessor

byoc_processor = ScriptProcessor(
    image_uri=ecr_image_uri,
    role=role, instance_count=1, instance_type="ml.c7i.xlarge",
    command=["python3"],
    env={"MLFLOW_APP_ARN": mlflow_app_arn},
    sagemaker_session=sm_session,
)

byoc_processor.run(
    code=os.path.join(CONTAINER_DIR, "evaluate.py"),
    inputs=[
        sagemaker.processing.ProcessingInput(source=MODEL_S3_URI, destination="/opt/ml/processing/model"),
        sagemaker.processing.ProcessingInput(source=f"s3://{DATASET_BUCKET}/{S3_TEST_PREFIX}", destination="/opt/ml/processing/test"),
    ],
    outputs=[
        sagemaker.processing.ProcessingOutput(output_name="evaluation", source="/opt/ml/processing/evaluation", destination=EVAL_S3_URI),
    ],
    wait=True, logs=True,
)
print(f"BYOC Processing job completed. Results saved to {EVAL_S3_URI}")

In [ ]:
eval_files = S3Downloader.list(EVAL_S3_URI)
for s3_uri in eval_files:
    if s3_uri.endswith("evaluation.json"):
        content = S3Downloader.read_file(s3_uri)
        print("BYOC Processing Job - Evaluation Metrics:")
        print(json.dumps(json.loads(content), indent=2))
        break
else:
    print("evaluation.json not found.")

---
## 11. SageMaker Pipeline

Train → Register → Evaluate の 3 ステップを SageMaker Pipeline として一括実行します。BYOC コンテナを使用します。

### DLC 版 Pipeline との違い

| 項目 | DLC 版 (`pytorch-pipeline.ipynb`) | BYOC 版 (このノートブック) |
|------|----------------------------------|---------------------------|
| Train Step | `PyTorch` Estimator | 汎用 `Estimator` + `image_uri` |
| Evaluate Step | `PyTorchProcessor` | `ScriptProcessor` + `image_uri` |
| コンテナ | AWS マネージド DLC | ECR 上のカスタムイメージ |
| 追加ライブラリ | `requirements.txt` で実行時インストール | Dockerfile でビルド時にインストール済み |

### PipelineSession について

`PipelineSession` は Pipeline 定義時に使用する特殊なセッションです。通常の `Session` と異なり、Estimator や Processor の `fit()` / `run()` を呼んでも実際のジョブは起動せず、Pipeline のステップ定義として記録されます。実際のジョブ実行は `pipeline.start()` 時に行われます。

In [ ]:
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import TrainingStep, ProcessingStep
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.inputs import TrainingInput
from sagemaker.workflow.pipeline_context import PipelineSession

pipeline_session = PipelineSession()
ROLE_ARN = outputs.get("SageMakerRoleArn", role)

dataset_bucket = DATASET_BUCKET
train_data_uri = f"s3://{dataset_bucket}/{S3_TRAIN_PREFIX}"
test_data_uri  = f"s3://{dataset_bucket}/{S3_TEST_PREFIX}"
model_output_uri = f"s3://{PROJECT_NAME}-model-{account_id}-{REGION}/output"
eval_output_uri  = f"s3://{PROJECT_NAME}-eval-{account_id}-{REGION}/output"

MODEL_GROUP_NAME = f"{PROJECT_NAME}-pytorch-byoc"
print(f"Role ARN:     {ROLE_ARN}")
print(f"Model group:  {MODEL_GROUP_NAME}")

### Train Step

BYOC コンテナを使った Training Job のステップを定義します。DLC 版と異なり、汎用の `Estimator` に `image_uri` でカスタムコンテナを指定します。

In [ ]:
pipeline_estimator = Estimator(
    image_uri=ecr_image_uri,
    entry_point="train.py",
    source_dir=CONTAINER_DIR,
    role=ROLE_ARN,
    instance_count=1,
    instance_type="ml.c7i.xlarge",
    output_path=model_output_uri,
    environment={"MLFLOW_APP_ARN": mlflow_app_arn, "MODEL_GROUP_NAME": MODEL_GROUP_NAME},
    metric_definitions=[
        {"Name": "train:accuracy", "Regex": r"Training accuracy: ([0-9.]+)"},
        {"Name": "train:f1", "Regex": r"Training f1: ([0-9.]+)"},
    ],
    sagemaker_session=pipeline_session,
)

train_step = TrainingStep(
    name="Train",
    estimator=pipeline_estimator,
    # input_mode: \"FastFile\" = S3 からオンデマンドストリーミング / \"File\" = 全量ダウンロード後に学習開始
    inputs={"train": TrainingInput(s3_data=train_data_uri, input_mode="FastFile")},
)
print("Train step created.")

### Register Step

Train Step で出力されたモデルアーティファクトを SageMaker Model Registry に登録します。`train_step.properties.ModelArtifacts.S3ModelArtifacts` でステップ間のデータ依存を定義しています。

In [ ]:
register_step = RegisterModel(
    name="RegisterModel",
    estimator=pipeline_estimator,
    model_data=train_step.properties.ModelArtifacts.S3ModelArtifacts,
    content_types=["application/json"],
    response_types=["application/json"],
    inference_instances=["ml.c7i.xlarge"],
    transform_instances=["ml.c7i.xlarge"],
    model_package_group_name=MODEL_GROUP_NAME,
)
print("Register step created.")

### Evaluate Step

BYOC コンテナを使った Processing Job で評価を実行します。`ScriptProcessor` に `image_uri` でカスタムコンテナを指定し、`evaluate.py` を実行します。

In [ ]:
pipeline_eval_processor = ScriptProcessor(
    image_uri=ecr_image_uri,
    role=ROLE_ARN, instance_count=1, instance_type="ml.c7i.xlarge",
    command=["python3"],
    env={"MLFLOW_APP_ARN": mlflow_app_arn},
    sagemaker_session=pipeline_session,
)

eval_args = pipeline_eval_processor.run(
    inputs=[
        sagemaker.processing.ProcessingInput(source=train_step.properties.ModelArtifacts.S3ModelArtifacts, destination="/opt/ml/processing/model"),
        sagemaker.processing.ProcessingInput(source=test_data_uri, destination="/opt/ml/processing/test"),
    ],
    outputs=[
        sagemaker.processing.ProcessingOutput(output_name="evaluation", source="/opt/ml/processing/evaluation", destination=eval_output_uri),
    ],
    # ScriptProcessor は source_dir をサポートしないため、
    # evaluate.py のパスを直接指定する (BYOC コンテナに依存ライブラリは含まれている)
    code=os.path.join(CONTAINER_DIR, "evaluate.py"),
)

eval_step = ProcessingStep(name="Evaluate", step_args=eval_args)
eval_step.add_depends_on([register_step])
print("Evaluate step created.")

### Pipeline の作成・実行

`upsert()` で Pipeline を SageMaker に登録し、`start()` で実行します。

In [ ]:
pipeline_name = f"{PROJECT_NAME}-pytorch-byoc-pipeline"

pipeline = Pipeline(
    name=pipeline_name,
    steps=[train_step, register_step, eval_step],
    sagemaker_session=pipeline_session,
)

pipeline.upsert(role_arn=ROLE_ARN)
print(f"Pipeline '{pipeline_name}' created/updated.")

In [ ]:
execution = pipeline.start()
print(f"Pipeline execution started: {execution.arn}")
print("Waiting for completion...")
execution.wait()
print(f"Pipeline execution status: {execution.describe()['PipelineExecutionStatus']}")

### 評価結果の確認

In [ ]:
eval_files = S3Downloader.list(eval_output_uri)
for s3_uri in eval_files:
    if s3_uri.endswith("evaluation.json"):
        content = S3Downloader.read_file(s3_uri)
        print("Pipeline Evaluation Metrics:")
        print(json.dumps(json.loads(content), indent=2))
        break
else:
    print("evaluation.json not found.")

---
## Pipeline 実行 (CLI、オプション)

上記の学習・評価を SageMaker Pipeline として一括実行することもできます。

```bash
# ターミナルから実行
./pipelines/scripts/run-pipeline.sh -c container-pytorch-dlc-byoc
```

In [ ]:
# Pipeline 実行 (CLI 経由)
# !cd .. && ./pipelines/scripts/run-pipeline.sh -c container-pytorch-dlc-byoc